In [ ]:
import kagglehub

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

# Download latest version
path = kagglehub.dataset_download("ourfuture/udmey-courses")

print("Path to dataset files:", path)

100%|██████████| 1.56M/1.56M [00:00<00:00, 25.6MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/ourfuture/udmey-courses/versions/1


In [ ]:
df = pd.read_excel(path + "/Udemy Courses.xlsx")

In [ ]:
df.insert(0, 'course_id', range(1, 1 + len(df)))

In [ ]:
df.columns = df.columns.str.lower().str.replace(' ', '_')
df.columns = df.columns.str.rstrip('_')

In [ ]:
def get_total_course_hours(course_hours_str):
  parts = course_hours_str.split()
  if not parts:
    return 0.0

  try:
    value = float(parts[0])
  except ValueError:
    # If the first part is not a number, return 0.0
    return 0.0

  # Use the 'in' operator for string containment check
  course_hours_str_lower = course_hours_str.lower()
  if "hour" in course_hours_str_lower: # Catches both "hour" and "hours"
    return value *60
  elif "min" in course_hours_str_lower: # Catches both "min" and "mins"
    return value
  else:
    # Default to hours if no specific unit is found. This might need adjustment
    # depending on the data.
    return value

df['total_course_hours'] = df['total_course_hours'].apply(get_total_course_hours)

In [ ]:
df.columns = df.columns.str.replace('total_course_hours', 'total_course_mins')

In [ ]:
mask = df['course_levels'].isna()
df.loc[mask, 'course_levels'] = df.loc[mask, 'total_no_of_lectures']

In [ ]:

df.loc[mask, 'total_no_of_lectures'] = "5"

In [ ]:
def get_total_lectures(lectures_str):
  parts = lectures_str.split()
  if not parts:
    return 0.0
  else:
    return float(parts[0])

df['total_no_of_lectures'] = df['total_no_of_lectures'].apply(get_total_lectures)

In [ ]:
df['total_reviews'] = df['total_reviews'].str.replace(' reviews', '')
df['total_reviews'] = df['total_reviews'].str.replace(' review', '')
df['total_reviews'] = df['total_reviews'].str.replace(' total hours', '')

In [ ]:
df['total_reviews'].astype(int)

,total_reviews
0,3907
1,1338
2,2242
3,173
4,1343
...,...
9963,85
9964,25
9965,40
9966,243


In [ ]:
df["ratings"] = df["ratings"].fillna(0)

In [ ]:
df.columns = df.columns.str.strip()
df['skills_you_gain'] = df['skills_you_gain'].fillna('')
# df['course_instructor_name'] = df['course_instructor_name'].fillna('')
# df['course_title'] = df['course_title'].fillna('')
# df['course_levels'] = df['course_levels'].fillna('')

In [ ]:
import pandas as pd
import spacy
import re

# Load spaCy's English model
nlp = spacy.load("en_core_web_sm")

In [ ]:
# 1. Create the Raw Soup (Title + Skills + Description/Level)
def create_soup(x):
    return f"{x['course_title']} {x['skills_you_gain']} {x['course_levels']}"

df['raw_text'] = df.apply(create_soup, axis=1)

In [ ]:
# 2. Define the Cleaning Function (Req 5)
def clean_text(text):
    # a. Convert to lowercase
    text = text.lower()

    # b. Remove special chars/HTML (Keep only letters and spaces)
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    # c. spaCy Processing (Tokenization + Lemmatization)
    # nlp(text) runs the neural pipeline
    doc = nlp(text)

    # Keep tokens that are NOT stop words and NOT punctuation
    # .lemma_ gives the root word (e.g., 'courses' -> 'course')
    clean_tokens = [token.lemma_ for token in doc if not token.is_stop and not token.is_punct]

    # Join back into a string
    return " ".join(clean_tokens)

print("Cleaning text with spaCy (this might take 2-3 minutes)...")
# We use a small sample first to test, then run on full df
# df['clean_text'] = df['raw_text'].apply(clean_text)

# OPTIMIZATION NOTE:
# Running .apply(clean_text) on 10,000 rows with spaCy is slow (~10 mins).
# For 10k rows, we can use nlp.pipe for speed:
clean_docs = list(nlp.pipe(df['raw_text'].tolist(), batch_size=50))
df['clean_text'] = [" ".join([t.lemma_ for t in doc if not t.is_stop and not t.is_punct]) for doc in clean_docs]

print("Data Cleaning Complete!")
print(df[['raw_text', 'clean_text']].head(1))

Cleaning text with spaCy (this might take 2-3 minutes)...
Data Cleaning Complete!
                                            raw_text  \
0  IT Fundamentals - Everything you need to know ...   

                                          clean_text  
0  Fundamentals need know Computer Skills great h...  


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pickle

# Req 6: TF-IDF with N-Grams (1, 2)
tfidf = TfidfVectorizer(ngram_range=(1, 2), stop_words='english')

# Fit on the CLEAN text
print("Vectorizing for TF-IDF...")
tfidf_matrix = tfidf.fit_transform(df['clean_text'])

# Save for Flask
with open('tfidf_matrix.pkl', 'wb') as f:
    pickle.dump(tfidf_matrix, f)
with open('vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf, f)

print("TF-IDF Models Saved.")

Vectorizing for TF-IDF...
TF-IDF Models Saved.


In [ ]:
from sentence_transformers import SentenceTransformer

# Req 6 & 9: Generate dense embeddings using sentence-transformers
model = SentenceTransformer('all-MiniLM-L6-v2')

print("Generating Neural Embeddings...")
# We use the CLEAN text here too, though BERT handles raw text well.
# Using clean text ensures consistency with Req 6.
nn_embeddings = model.encode(df['clean_text'].tolist(), show_progress_bar=True)

# Save for Flask
with open('nn_embeddings.pkl', 'wb') as f:
    pickle.dump(nn_embeddings, f)

print("Neural Models Saved.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Generating Neural Embeddings...


Batches:   0%|          | 0/312 [00:00<?, ?it/s]

Neural Models Saved.


In [ ]:
with open('course_list.pkl', 'wb') as f:
    pickle.dump(df, f)
print("Dataframe Saved.")

Dataframe Saved.


In [ ]:
with open('dataset.csv', 'w') as f:
    df.to_csv(f)
print("Dataframe Saved.")

Dataframe Saved.


In [ ]:
df.head()

,course_id,course_title,course_instructor_name,skills_you_gain,total_course_mins,total_no_of_lectures,ratings,total_reviews,course_levels,course_links,course_thumbnail_image,raw_text,clean_text
0,1,IT Fundamentals - Everything you need to know ...,Imran Afzal,Computer Skills -- Great help for passing Comp...,690.0,151.0,4.6,3907,All Levels,https://www.udemy.com/course/it-fundamentals-e...,https://img-c.udemycdn.com/course/240x135/3627...,IT Fundamentals - Everything you need to know ...,Fundamentals need know Computer Skills great h...
1,2,IT Support Technical Skills Bootcamp,Jobskillshare Community,Hands-on Technical skills for Support Profess...,2310.0,75.0,4.4,1338,Beginner,https://www.udemy.com/course/it-support-techni...,https://img-b.udemycdn.com/course/240x135/4663...,IT Support Technical Skills Bootcamp Hands-on ...,Support Technical Skills Bootcamp Hands Techni...
2,3,IT for beginners | IT for dummies | IT for non-IT,Maaike van Putten,Everything all non-technical professionals in ...,90.0,32.0,4.3,2242,Beginner,https://www.udemy.com/course/it-for-beginners/,https://img-b.udemycdn.com/course/240x135/1976...,IT for beginners | IT for dummies | IT for non...,beginner | dummy | non non technical professio...
3,4,IT Support Fundamentals for IT Beginners,Jobskillshare Community,Support Skills for Beginners,900.0,53.0,4.4,173,Beginner,https://www.udemy.com/course/it-support-fundam...,https://img-c.udemycdn.com/course/240x135/4714...,IT Support Fundamentals for IT Beginners Suppo...,Support Fundamentals Beginners Support Skills ...
4,5,Desktop IT Support Level 1 & 2 in real life (T...,Tareq Tech,Learn everything in real life Troubleshooting ...,450.0,60.0,4.1,1343,Beginner,https://www.udemy.com/course/desktop-support/,https://img-c.udemycdn.com/course/240x135/2511...,Desktop IT Support Level 1 & 2 in real life (T...,desktop Support Level 1 2 real life troublesho...


In [ ]:

with open('datasethead.json', 'w') as f:
  df.to_json(f)
print("Dataframe Saved.")


Dataframe Saved.


In [ ]:
df.T.head(200)

,0,1,2,3,4,5,6,7,8,9,...,9958,9959,9960,9961,9962,9963,9964,9965,9966,9967
course_id,1,2,3,4,5,6,7,8,9,10,...,9959,9960,9961,9962,9963,9964,9965,9966,9967,9968
course_title,IT Fundamentals - Everything you need to know ...,IT Support Technical Skills Bootcamp,IT for beginners | IT for dummies | IT for non-IT,IT Support Fundamentals for IT Beginners,Desktop IT Support Level 1 & 2 in real life (T...,CompTIA IT Fundamentals (ITF+) Complete Course...,Learning IT Help Desk for Beginners,Complete Linux Training Course to Get Your Dre...,TOTAL: CompTIA IT Fundamentals ITF+ (FCO-U61).,"CompTIA Mastery: A+, Network+, Security+ Ultim...",...,I Can Do All Things,How to Think for Yourself,3 STEPS TO RAISING CAPITAL FAST [INTRO],Getting Started With TestProject,Personal interview made easy with tricks & tac...,Interview Skills to Crack the Interview | Inte...,Exercise with Aromatherapy,How to Buy Cloud – Strategies for Cloud Procur...,PLM Basics Course,The Essential Beginners Guitar Course
course_instructor_name,Imran Afzal,Jobskillshare Community,Maaike van Putten,Jobskillshare Community,Tareq Tech,"Mike Chapple, Ph.D.",Emilio Aguero,Imran Afzal,"Total Seminars • Over 1 Million Enrollments, S...","PaceIT Academy, M. Fahmid Chowdhury",...,Clarence W. Fell,Steve Churchill,"Finance for Entrepreneurs, Lili Balfour",Dave Westerveld,Vinay Modi,B Yadgiri,瀬口 芳美,"Amazon Web Services (AWS), Blaine Sundrud",Helena Gutierrez,Jon Varley
skills_you_gain,Computer Skills -- Great help for passing Comp...,Hands-on Technical skills for Support Profess...,Everything all non-technical professionals in ...,Support Skills for Beginners,Learn everything in real life Troubleshooting ...,Everything you need to know to pass the CompTI...,Must know skills for Helpdesk and Service Desk...,The BEST Linux Administration course that prep...,What EVERY USER needs to know about basic con...,"ALL in ONE : CompTIA A+, CompTIA Network+. Com...",...,Moving from Abstract Theory to Daily Reality,"Increase your intellectual independence, self-...",This is a brief introduction on using public p...,Learn to use a powerful free test automation tool,"'""Cracking interview is like playing a game of...",Learn how to Answer commonly asked Interview Q...,Massage and stretch in the aroma bath,Understand the process to start and embrace di...,Build your Product Lifecycle Management knowle...,Easy to Follow Format that will have you playi...
total_course_mins,690.0,2310.0,90.0,900.0,450.0,240.0,270.0,2130.0,300.0,1380.0,...,60.0,120.0,60.0,60.0,90.0,35.0,37.0,120.0,60.0,60.0
total_no_of_lectures,151.0,75.0,32.0,53.0,60.0,69.0,27.0,257.0,64.0,183.0,...,12.0,25.0,8.0,11.0,19.0,7.0,31.0,37.0,15.0,33.0
ratings,4.6,4.4,4.3,4.4,4.1,4.7,4.5,4.6,4.6,4.3,...,4.2,4.2,4.2,4.5,4.2,4.6,4.6,4.6,4.3,4.2
total_reviews,3907,1338,2242,173,1343,1330,777,26868,10243,332,...,19,39,340,131,71,85,25,40,243,14
course_levels,All Levels,Beginner,Beginner,Beginner,Beginner,Beginner,All Levels,All Levels,Beginner,All Levels,...,All Levels,All Levels,All Levels,Beginner,All Levels,All Levels,All Levels,Beginner,Beginner,Beginner
course_links,https://www.udemy.com/course/it-fundamentals-e...,https://www.udemy.com/course/it-support-techni...,https://www.udemy.com/course/it-for-beginners/,https://www.udemy.com/course/it-support-fundam...,https://www.udemy.com/course/desktop-support/,https://www.udemy.com/course/certmike-comptia-...,https://www.udemy.com/course/helpdesk-pro/,https://www.udemy.com/course/complete-linux-tr...,https://www.udemy.com/course/it-fundamentals-f...,https://www.udemy.com/course/computer-network-...,...,https://www.udemy.com/course/i-can-do-all-things/,https://www.udemy.com/course/think-for-yourself/,https://www.udemy.com/course/3stepsintro/,https://www.udemy.com/course/getting-started-w...,https://www.udemy.com/course/personal-intervie...,https://www.udemy.com/course/how-to-answer-mos...,https://www.udemy.com/course/exercise-with-aro...,https://www.udemy.com/course/aws-how-to-buy-cl...,https://www